# Duplicate Removal: Part B (Grape, Peach, Pepper, Potato, Cherry, Squash, Strawberry, Blueberry)

## Setup

In [10]:
import csv
import hashlib
import shutil
from collections import Counter
from pathlib import Path

import imagehash
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from tqdm import tqdm

SOURCE_ROOT = Path("/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color")
WORK_ROOT = Path("/kaggle/working/color")
SELECTED_CROPS = ["Grape", "Peach", "Pepper", "Potato", "Cherry", "Squash", "Strawberry", "Blueberry"]
EXPECTED_IMAGE_COUNT = 18154
PART_NAME = "part_b"
HAMMING_THRESHOLD = 4
CONFLICT_RULE = "keep_one"
DRY_RUN = False
WRITE_DELETE_LOG = True
START_FRESH = True
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

BIT_COUNT_TABLE = np.array([bin(value).count("1") for value in range(256)], dtype=np.uint8)


## Functions

In [11]:
def crop_name(folder_name: str) -> str:
    return folder_name.split("___")[0]


def matches_crop(crop: str, selected: str) -> bool:
    return crop == selected or crop.startswith(selected + "_") or crop.startswith(selected + ",")


def is_selected_crop(folder_name: str) -> bool:
    crop = crop_name(folder_name)
    return any(matches_crop(crop, selected) for selected in SELECTED_CROPS)


def copy_selected_crops() -> None:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    for folder in sorted(SOURCE_ROOT.iterdir()):
        if not folder.is_dir() or not is_selected_crop(folder.name):
            continue
        destination = WORK_ROOT / folder.name
        if destination.exists():
            print(f"Already copied: {folder.name}")
            continue
        print(f"Copying {folder.name}")
        shutil.copytree(folder, destination)


def list_image_files() -> list:
    files = [
        path for path in WORK_ROOT.rglob("*")
        if path.suffix.lower() in IMAGE_EXTENSIONS and is_selected_crop(path.parent.name)
    ]
    return sorted(files)


def compute_md5(path: Path) -> str:
    digest = hashlib.md5()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def build_records(files: list) -> list:
    records = []
    for path in tqdm(files, desc="Hashing"):
        try:
            with Image.open(path) as image:
                width, height = image.size
                perceptual_hash = int(str(imagehash.phash(image.convert("RGB"))), 16)
            records.append({
                "path": str(path.relative_to(WORK_ROOT.parent)),
                "crop": crop_name(path.parent.name),
                "class": path.parent.name,
                "md5": compute_md5(path),
                "phash": perceptual_hash,
                "width": width,
                "height": height,
            })
        except Exception as error:
            print(f"Skipped unreadable file {path}: {error}")
    return records


def count_differing_bits(values: np.ndarray) -> np.ndarray:
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(values)
    return BIT_COUNT_TABLE[values.view(np.uint8).reshape(-1, 8)].sum(axis=1)


def group_duplicates(records: list) -> list:
    count = len(records)
    hashes = np.array([record["phash"] for record in records], dtype=np.uint64)
    processing_order = sorted(
        range(count),
        key=lambda i: (-(records[i]["width"] * records[i]["height"]), records[i]["path"]),
    )

    keeper_hashes = np.zeros(count, dtype=np.uint64)
    groups = []
    for index in tqdm(processing_order, desc="Comparing"):
        keeper_count = len(groups)
        if keeper_count:
            distances = count_differing_bits(keeper_hashes[:keeper_count] ^ hashes[index])
            nearest = int(np.argmin(distances))
            if distances[nearest] <= HAMMING_THRESHOLD:
                groups[nearest].append(index)
                continue
        keeper_hashes[keeper_count] = hashes[index]
        groups.append([index])
    return groups


def select_deletions(records: list, groups: list) -> dict:
    deletions = {}
    for group in groups:
        if len(group) == 1:
            continue
        classes = {records[i]["class"] for i in group}
        has_conflict = len(classes) > 1
        if has_conflict and CONFLICT_RULE == "drop_all":
            for i in group:
                deletions[i] = "label conflict (all copies removed)"
        else:
            reason = "label conflict (best copy kept)" if has_conflict else "duplicate"
            for i in group[1:]:
                deletions[i] = reason
    return deletions


def write_csv(path: str, header: list, rows: list) -> None:
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(rows)


def main() -> None:
    if CONFLICT_RULE not in ("drop_all", "keep_one"):
        raise ValueError("CONFLICT_RULE must be 'drop_all' or 'keep_one'")

    if not SELECTED_CROPS:
        available = sorted({crop_name(folder.name) for folder in SOURCE_ROOT.iterdir() if folder.is_dir()})
        print("SELECTED_CROPS is empty. Crops available in the dataset:", available)
        return

    if START_FRESH and WORK_ROOT.exists():
        shutil.rmtree(WORK_ROOT)

    copy_selected_crops()
    files = list_image_files()
    print(f"Images found for {SELECTED_CROPS}: {len(files)} (expected {EXPECTED_IMAGE_COUNT})")

    per_crop = Counter(crop_name(path.parent.name) for path in files)
    for selected in SELECTED_CROPS:
        total = sum(count for crop, count in per_crop.items() if matches_crop(crop, selected))
        flag = "   <-- nothing found, check the name" if total == 0 else ""
        print(f"  {selected}: {total}{flag}")

    records = build_records(files)
    if not records:
        raise SystemExit("No readable images found. Check SOURCE_ROOT and SELECTED_CROPS.")

    groups = group_duplicates(records)
    deletions = select_deletions(records, groups)

    for index in tqdm(sorted(deletions), desc="Dry run" if DRY_RUN else "Deleting"):
        if not DRY_RUN:
            (WORK_ROOT.parent / records[index]["path"]).unlink(missing_ok=True)

    remaining = [record for i, record in enumerate(records) if i not in deletions]
    columns = list(records[0].keys())
    hash_file = Path(f"{PART_NAME}_hashes.csv")
    write_csv(str(hash_file), columns, [[r[c] for c in columns] for r in remaining])

    if WRITE_DELETE_LOG:
        write_csv(
            f"{PART_NAME}_deleted_log.csv",
            ["path", "reason"],
            [[records[i]["path"], deletions[i]] for i in sorted(deletions)],
        )

    print("\nSummary")
    print(f"Before: {len(records)} | Deleted: {len(deletions)} | After: {len(remaining)}")
    for reason, total in Counter(deletions.values()).items():
        print(f"  {reason}: {total}")
    if DRY_RUN:
        print("Dry run: no files were deleted.")
    print(f"Saved: {hash_file.resolve()}")


## Run

In [12]:
main()


Copying Blueberry___healthy
Copying Cherry_(including_sour)___Powdery_mildew
Copying Cherry_(including_sour)___healthy
Copying Grape___Black_rot
Copying Grape___Esca_(Black_Measles)
Copying Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
Copying Grape___healthy
Copying Peach___Bacterial_spot
Copying Peach___healthy
Copying Pepper,_bell___Bacterial_spot
Copying Pepper,_bell___healthy
Copying Potato___Early_blight
Copying Potato___Late_blight
Copying Potato___healthy
Copying Squash___Powdery_mildew
Copying Strawberry___Leaf_scorch
Copying Strawberry___healthy
Images found for ['Grape', 'Peach', 'Pepper', 'Potato', 'Cherry', 'Squash', 'Strawberry', 'Blueberry']: 18154 (expected 18154)
  Grape: 4062
  Peach: 2657
  Pepper: 2475
  Potato: 2152
  Cherry: 1906
  Squash: 1835
  Strawberry: 1565
  Blueberry: 1502


Deleting: 100%|██████████| 70/70 [00:00<00:00, 17678.30it/s]



Summary
Before: 18154 | Deleted: 70 | After: 18084
  duplicate: 62
  label conflict (best copy kept): 8
Saved: /kaggle/working/part_b_hashes.csv


## Deleted Images

In [13]:
deleted_log = pd.read_csv(f"{PART_NAME}_deleted_log.csv")
display(deleted_log["reason"].value_counts().rename("images").to_frame())
deleted_log.head(10)


,images
reason,
duplicate,62
label conflict (best copy kept),8


,path,reason
0,color/Blueberry___healthy/3b4ab949-7b1d-4127-a...,duplicate
1,color/Blueberry___healthy/76610fbe-b5da-460e-a...,duplicate
2,color/Blueberry___healthy/7b7ada95-0a08-4b41-a...,duplicate
3,color/Blueberry___healthy/8c220aee-d920-4f0a-9...,duplicate
4,color/Blueberry___healthy/a69c62a5-dcb0-4473-a...,duplicate
5,color/Blueberry___healthy/db4a99f7-55a2-4cbb-8...,label conflict (best copy kept)
6,color/Blueberry___healthy/dc1d0041-ddd3-4bd4-b...,duplicate
7,color/Blueberry___healthy/eee48720-b1ef-499f-9...,duplicate
8,color/Blueberry___healthy/fe4de3f6-ef8e-4e3f-b...,duplicate
9,color/Cherry_(including_sour)___Powdery_mildew...,duplicate


## Unique Images

In [14]:
unique_images = pd.read_csv(f"{PART_NAME}_hashes.csv")
files_on_disk = {str(path.relative_to(WORK_ROOT.parent)) for path in list_image_files()}

print(f"Unique images listed: {len(unique_images)}")
if DRY_RUN:
    print("Dry run: file check skipped")
else:
    print(f"Matches the files left on disk: {set(unique_images['path']) == files_on_disk}")
display(unique_images["class"].value_counts().rename("images").to_frame())
unique_images.head(10)


Unique images listed: 18084
Matches the files left on disk: True


,images
class,
Peach___Bacterial_spot,2296
Squash___Powdery_mildew,1834
Blueberry___healthy,1493
"Pepper,_bell___healthy",1464
Grape___Esca_(Black_Measles),1380
Grape___Black_rot,1179
Strawberry___Leaf_scorch,1106
Grape___Leaf_blight_(Isariopsis_Leaf_Spot),1076
Cherry_(including_sour)___Powdery_mildew,1040


,path,crop,class,md5,phash,width,height
0,color/Blueberry___healthy/008c85d0-a954-4127-b...,Blueberry,Blueberry___healthy,cf578d6a69f6e52c5dbc35c765c579a1,16245502320671202104,256,256
1,color/Blueberry___healthy/00fee259-67b7-4dd7-8...,Blueberry,Blueberry___healthy,b41098761ac25720ef74205cafab762c,18137548530247295666,256,256
2,color/Blueberry___healthy/011405e4-a92a-44b6-9...,Blueberry,Blueberry___healthy,b1f3b1ddba85c94f48b030370b27e27b,16804221877173732812,256,256
3,color/Blueberry___healthy/0137389a-9b78-4a94-a...,Blueberry,Blueberry___healthy,fde3473efb11fc93021e8b296b8d33e0,13022945322765542289,256,256
4,color/Blueberry___healthy/013ffdc5-cbd1-4202-b...,Blueberry,Blueberry___healthy,85246e4d444546a8ae59d33d97c9ddb9,12640426104887803012,256,256
5,color/Blueberry___healthy/01ca7a9f-80a6-4693-a...,Blueberry,Blueberry___healthy,baeb84a799716ebccf07d983bcb8990e,13771940455951132048,256,256
6,color/Blueberry___healthy/02159532-6430-4006-a...,Blueberry,Blueberry___healthy,b96d778aeb63b5f1fda61eb7da757bdc,16457441420364205720,256,256
7,color/Blueberry___healthy/021e662c-f3a0-4e08-a...,Blueberry,Blueberry___healthy,281a6a43111c2c9c401a9fb4e426121a,13393924162730427191,256,256
8,color/Blueberry___healthy/022c53af-7964-4692-8...,Blueberry,Blueberry___healthy,87659af0b89f06c3a988b77f88867766,12734151996627248795,256,256
9,color/Blueberry___healthy/0257329a-7b6b-42a5-9...,Blueberry,Blueberry___healthy,065137eeb3044fc1150d3b98835b69c6,16917390549532155497,256,256
